# Word features in LLM SD

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
data = pd.read_csv("../inference_results/results_exp1/concatenated_results.csv", low_memory=False)

In [ ]:
PROMPT_LIST_DF = pd.read_excel("./dataset/prompt_list.xlsx", header=0)
WORD_FEATURES_DF = pd.read_excel("./dataset/word_features_exp1_with_sd.xlsx",
                                 sheet_name="data", index_col=None, header=[0])

In [ ]:
all_results_df = []

num_of_iterations = 10
model_name_list = [
  "gpt-4o-2024-05-13",
  "gpt-4o-mini-2024-07-18",
  "meta-llama_Meta-Llama-3.1-70B-Instruct",
  "meta-llama_Meta-Llama-3.1-405B-Instruct",
]

results_df = WORD_FEATURES_DF.copy()

for model_name in model_name_list:
  for feature_type in PROMPT_LIST_DF.dropna().feature_type:
    columns_to_analyze = [f'{model_name}_{feature_type}_{i}' for i in range(num_of_iterations)]
    existing_columns = [col for col in columns_to_analyze if col in data.columns]

    data_numeric = data[existing_columns]
    std = data_numeric.apply(pd.to_numeric, errors='coerce').std(axis=1, ddof=1)

    std.name = f"{model_name}_{feature_type}_sd"
    results_df.loc[:, std.name] = std

results_df.to_csv('./results_sd/std.csv', na_rep='NA')

### sd mean

In [ ]:
std_mean_list_all = []
# llm
for model_name in model_name_list:
  std_mean_list = []
  for feature_type in PROMPT_LIST_DF.dropna().feature_type:
    column_name = f"{model_name}_{feature_type}_sd"
    std = results_df[column_name]
    std_mean_list.append(std.mean())

    # Print the average std for the current feature type
    # print(f"Average std for {column_name}:{std.mean().round(3)}")
  std_mean_list_all.append(std_mean_list)

# human
std_mean_list = []
for feature_type in PROMPT_LIST_DF.dropna().feature_type:
  column_name = f"{feature_type}_sd"
  std = results_df[column_name]
  std_mean_list.append(std.mean())

  # Print the average std for the current feature type
  # print(f"Average std for {column_name}:{std.mean().round(3)}")
std_mean_list_all.append(std_mean_list)

In [ ]:
std_mean_df = pd.DataFrame(std_mean_list_all)
std_mean_df.columns = [feature_type for feature_type in PROMPT_LIST_DF.dropna().feature_type]
std_mean_df.index = model_name_list + ["human"]
std_mean_df

std_mean_df.to_csv('./results_sd/std_mean.csv', na_rep='NA')

In [ ]:
std_mean_df

### sd cossim

In [ ]:
std_cossim_list_all = []
# llm
for model_name in model_name_list:
  std_cossim_list = []
  for feature_type in PROMPT_LIST_DF.dropna().feature_type:
    if feature_type == "babiness":
      continue
    column_name = f"{model_name}_{feature_type}_sd"
    std_llm = results_df[column_name]

    column_name = f"{feature_type}_sd"
    std_human = results_df[column_name]

    # tmp_df = pd.concat([std_llm, std_human], axis=1).dropna()
    # tmp_df.plot.scatter(x=tmp_df.columns[0], y=tmp_df.columns[1])

    cos_sim = cosine_similarity(
      pd.concat([std_llm, std_human], axis=1).dropna().T
    )
    std_cossim_list.append(cos_sim[0][1])

  std_cossim_list_all.append(std_cossim_list)

In [ ]:
std_cossim_df = pd.DataFrame(std_cossim_list_all)
std_cossim_df.columns = [feature_type for feature_type in PROMPT_LIST_DF.dropna().feature_type][:-1]
std_cossim_df.index = model_name_list
std_cossim_df

std_cossim_df.to_csv('./results_sd/std_cossim.csv', na_rep='NA')


In [ ]:
std_cossim_df